In [1]:
import os
os.environ['KAGGLE_USERNAME'] = "xxxxxx"
os.environ['KAGGLE_KEY'] = "xxxxxx"
!kaggle datasets download -d cornell-university/arxiv
!unzip arxiv.zip

Dataset URL: https://www.kaggle.com/datasets/cornell-university/arxiv
License(s): CC0-1.0
100% 1.58G/1.58G [00:15<00:00, 98.3MB/s]
100% 1.58G/1.58G [00:15<00:00, 109MB/s] 
Archive:  arxiv.zip
  inflating: arxiv-metadata-oai-snapshot.json  


Parsing e building the graph (nodes = authors, arcs = collaborations)

In [2]:
from pyspark.sql import SparkSession
import json

#Creating a Spark session
spark = SparkSession.builder.appName("ArxivPageRank").getOrCreate()
sc = spark.sparkContext

#Dataset uploaded in a RDD (Resilient Distributed Dataset)
#Collection of elements distributed on more node on a cluster (//)
ds = sc.textFile("arxiv-metadata-oai-snapshot.json")

ds = ds.sample(False, 0.01, 555).cache()
#ds = ds.sample(False, 1, 555).cache()
#with 0.01 = faction of the sample and 555 = seed

#Parsing (id, authors_parsed, categories)
def parse_record(line):
  try:
    record = json.loads(line)
    paper_id = record.get("id")
    #The var authors contains "authors_parsed" if present, otherwise an empty list
    authors = record.get("authors_parsed", [])
    categories = record.get("categories", "") #Required for the version with categories

    #Trasforming the authors_parsed list in a list of readable names
    #union of name and surname in a single line
    author_names = [
        f"{a[0]} {a[1]}".strip().lower()
        #for: iterating on the authors
        for a in authors if len(a) >= 2 ] #if in case of incomplete format

    #Removing empty strings or really short names (deleting results like a b)
    author_names = [name for name in author_names if len(name) > 1]

    if paper_id and author_names and categories:
        #(ID, list_of_authors, categories_string)
        return (paper_id, author_names, categories)
    else:
        return None
  except:
        return None

#New RDD starting from the raw one
papers_rdd = (
    ds
    .map(parse_record) #Applying the parse_record function to each line of ds
    .filter(lambda x: x is not None) #Keeping only the valid elements
)

#[(paper_id, [author1, author2, ...], category),
 #(paper_id, [author1, author2, ...], category), ...]
#Example: [('0704.0004', ['callan david'], 'math.CO'), ...]

for x in papers_rdd.take(5): print(x)

('0704.0076', ['gronau michael'], 'hep-ph hep-ex')
('0704.0208', ['hagge tobias j.', 'hong seung-moon'], 'math.GT math.QA')
('0704.0230', ['duffard r.', 'roig f.'], 'astro-ph')
('0704.0250', ['pankov s.', 'dobrosavljevic v.'], 'cond-mat.str-el cond-mat.dis-nn')
('0704.0354', ['podolsky d.'], 'hep-th gr-qc')


Co-authorship graph

In [3]:
#Function to generate all possible pairs of co-authors from a single paper
def generate_coauthors_pairs(data):
    #Ignoring the third element (category) because it is not necessary in this function
    paper_id, authors = data ##
    #At least 2 authors for each paper to create a co-authorship link
    #Shortening the time avoiding using the hyper-authored papers
    if len(authors) < 2 or len(authors) > 30:
        return []

    #Using permutations to create directed edges (A -> B and B -> A)
    #Source -> receiver
    edges = []
    #Nested loops to create all directed permutations (A -> B and B -> A)
    for i in range(len(authors)):
        for j in range(len(authors)):
            #Avoiding author with themselves
            if i != j:
                edges.append((authors[i], authors[j])) # =>list of tuples
    return edges

    #Different solution: using the itertools library instead of the two for loops:
    #import itertools
    #return list(itertools.permutations(authors, 2))

#We have an RDD with three categories but we only need two (ID, list_of_authors) for this part
paper_author_rdd = papers_rdd.map(lambda x: (x[0], x[1])) ## x[0] = paper_id, x[1] = author_names (list)

#Flattening the RDD into an RDD of author pairs
#Each element will be (author_source, author_destination)
edges_rdd = paper_author_rdd.flatMap(generate_coauthors_pairs)

#Creating the adjacency list ##
adj_list = (
    edges_rdd
    .mapValues(lambda x: [x])          #From (A, B) to (A, [B])
    .reduceByKey(lambda a, b: list(set(a + b)))
    .cache() #The graph structure is reused in every iteration
)

#Debug: printing the first 3 entries of the adjacency list
print("Adjacency List (Author -> [Collaborators]):")
for author, collaborators in adj_list.take(3):
    print(f"{author} collaborated with: {collaborators}")


Adjacency List (Author -> [Collaborators]):
sun z. y. collaborated with: ['tsang m. b.', 'niikura m.', 'takeshita e.', 'wallace m. s.', 'motobayashi t.', 'hui h.', 'takeuchi s.', 'cook j.', 'lynch w. g.', 'mocko m.', 'sakurai h.', 'aoi n.', 'delaunay f.', 'onishi t.', 'iwasaki h.', 'rogers a. m.', 'famiano m. a.', 'imai n.', 'suzuki h.', 'stolz a.']
muller p. collaborated with: ['ranguelov b.', 'metois j. j.']
chevassus nicolas collaborated with: ['chablat damien', 'merlhiot xavier', 'guillaume françois', 'rennuit antoine', 'andriot claude', 'chedmail patrick', 'micaelli alain']


Inizializing the ranks: creation of an RDD in which all the authors have a starting score of 1.0 (starting point for the iterations in PageRank)

In [4]:
#Initialize each author's rank to 1.0
#Using the keys (authors) from the adjacency list and assigning them the starting value
ranks_rdd = adj_list.mapValues(lambda v: 1.0)

Function that is going to be used in the PageRank for loop

In [5]:
#Function to compute and sum contributions (inside the loop)
def compute_contributions(author_data):
    #author_data is (author_name, (collaborators_list, current_rank))
    author, (collaborators, rank) = author_data
    num_collaborators = len(collaborators)

    #Each collaborator receives a fraction of the current author's rank
    for collaborator in collaborators:
        yield (collaborator, rank / num_collaborators)

For loop to update the ranks:

In [6]:
#PageRank Iterative Loop
num_iterations = 5
damping_factor = 0.85

for i in range(num_iterations):
    #Join ranks with adjacency list ##
    #Result: (author, (collaborators_list, current_rank))
    joined_rdd = adj_list.join(ranks_rdd)

    #Distribute rank to collaborators
    #compute_contributions returns (collaborator, contribution)
    contributions = joined_rdd.flatMap(compute_contributions)

    #Sum all contributions for each author
    summed_contributions = contributions.reduceByKey(lambda x, y: x + y)

    #Application of the Damping Factor formula
    # PR(A) = (1 - d) + d * (sum of contributions)
    ranks_rdd = summed_contributions.mapValues(
        lambda rank_sum: (1 - damping_factor) + damping_factor * rank_sum
    )

    print(f"Completed iteration {i + 1}")

#Cache the final result
ranks_rdd.cache()

Completed iteration 1


PythonRDD[17] at RDD at PythonRDD.scala:56

Sorting the results and visualizing the first results

In [7]:
#Sort the ranks RDD by the rank value (index 1 of the tuple)
#ascending=False -> the highest ranks first
top_authors_rdd = ranks_rdd.sortBy(lambda x: x[1], ascending=False)

#Take the top 10 results and print them
#.title() to make the names look better (ex. 'john doe' -> 'John Doe') ##
print("Top 10 authors")

for author, rank in top_authors_rdd.take(10):
    print(f"Author: {author.title():<25} | Score: {rank:.4f}")

Top 10 authors
Author: Liu Yang                  | Score: 20.1767
Author: Zhang Wei                 | Score: 15.0356
Author: Zhang Yu                  | Score: 14.4001
Author: Watanabe Kenji            | Score: 13.5392
Author: Taniguchi Takashi         | Score: 13.5392
Author: Wang Wei                  | Score: 12.4331
Author: Li Wei                    | Score: 11.1001
Author: Wang Xin                  | Score: 10.9509
Author: Chen Xi                   | Score: 10.8067
Author: Wang Jun                  | Score: 10.4652


In [8]:
adj_list.unpersist()

PythonRDD[8] at RDD at PythonRDD.scala:56

DIFFERENT VERSIONS
1. Weighted PageRank: how many papers written together (more collaborations, stronger score)
2. Top authors for category: most important authors for specific sectors



1. Weighted PageRank: count the occurrences of each couple. In the PageRank loop, each score is weighed basing on the number of collaborations (insted of being divided equally between the authors as before).

In [9]:
def compute_weighted_contributions(author_data):
    #author_data: (author_name, (weighted_collaborators_list, current_rank))
    author, (weighted_collabs, rank) = author_data

    #Calculate the sum of all weights for this author
    total_weight = sum(weight for collab, weight in weighted_collabs)

    for collaborator, weight in weighted_collabs:
        #Each collaborator gets a share proportional to the weight of the link
        contribution = rank * (weight / total_weight)
        yield (collaborator, contribution)

def run_pagerank_weighted(input_papers_rdd, iterations = 5, damping=0.85):
    #Build weighted adjacency list
    weighted_edges = (
        input_papers_rdd
        .flatMap(generate_coauthors_pairs)
        #((author1, author2), 1)
        .map(lambda pair: (pair, 1))
        #((author1, author2), sum)
        .reduceByKey(lambda a, b: a + b)
    )

    adj_list = (
    weighted_edges
    #Creating a list containing the tuple
    #Risult:(autore_A, [(autore_B, peso)])
    .map(lambda x: (x[0][0], [(x[0][1], x[1])]))
    #Concatenating the lists of the same authors
    .reduceByKey(lambda a, b: a + b)
    .cache()
)

    #Initialize ranks to 1.0
    ranks = adj_list.mapValues(lambda v: 1.0)

    #Iterative Loop
    for i in range(iterations):
        #Joining adjacency list with current ranks
        joined_rdd = adj_list.join(ranks)
        #Calculating contributions based on weights
        contributions = joined_rdd.flatMap(compute_weighted_contributions)
        #Summing the contributions
        summed_contributions = contributions.reduceByKey(lambda x, y: x + y)

        #PageRank formula with damping factor
        ranks = summed_contributions.mapValues(
            lambda r_sum: (1 - damping) + damping * r_sum
        )

    #Final RDD sorted by rank:
    return ranks.sortBy(lambda x: x[1], ascending=False)

In [10]:
#Isolating ID of the paper and the list of authors
weighted_authors = run_pagerank_weighted(paper_author_rdd)

for author, rank in weighted_authors.take(10):
    print(f"Author: {author.title():<25} | Score: {rank:.4f}")

Author: Liu Yang                  | Score: 20.6193
Author: Zhang Wei                 | Score: 15.0253
Author: Zhang Yu                  | Score: 14.3863
Author: Watanabe Kenji            | Score: 13.8208
Author: Taniguchi Takashi         | Score: 13.8208
Author: Wang Wei                  | Score: 12.4997
Author: Li Wei                    | Score: 11.1199
Author: Wang Xin                  | Score: 11.0770
Author: Chen Xi                   | Score: 10.8163
Author: Wang Jun                  | Score: 10.5776


2. Top authors for category (using the categories part of the dataset). The nodes are still the authors.
The RDD is filtered basing on a category (or group of them) before building the graph, to obtain the most important authors for specific sectors.

In [11]:
#Filtering the papers to include only the papers belonging to one category
cat_papers_rdd = papers_rdd.filter(lambda x: "hep-ph" in x[2]).map(lambda x: (x[0], x[1]))
#Map keeps only (id, authors) for the PageRank function
#Filter using x[2] that is the category

cat_authors = run_pagerank_weighted(cat_papers_rdd)

for author, rank in cat_authors.take(10):
    print(f"Author: {author.title():<25} | Score: {rank:.4f}")

Author: Frank Mariana             | Score: 4.3292
Author: Kobayashi Tatsuo          | Score: 4.1096
Author: Li Tianjun                | Score: 4.0458
Author: Han Tao                   | Score: 3.7787
Author: Spannowsky Michael        | Score: 3.5671
Author: Azizi K.                  | Score: 3.5500
Author: Martins C. J. A. P.       | Score: 3.5500
Author: He Xiao-Gang              | Score: 3.3729
Author: Barger Vernon             | Score: 3.3686
Author: Kim C. S.                 | Score: 3.3439
